# 🎬 Text-to-Video Part A: Compression & Temporal Architecture

## Why Video is Hard

Think of video as a **flipbook**: each page is a still image, and flipping through them creates the illusion of motion. The problem is that video files are *enormous*.

**The math:**
```
120 frames × 1280 × 720 pixels × 3 channels = 110,592,000 ≈ 110.6M values
```
A single 5-second clip at 24fps already has ~110M pixel values. Training a diffusion model directly on this raw space is computationally infeasible.

**Solution: Latent Diffusion Models (LDM)**

Instead of operating on raw pixels, we:
1. **Encode** video frames into a compact latent space with a VAE (e.g., 8× spatial compression)
2. **Run diffusion** in that smaller latent space
3. **Decode** back to pixels only at inference time

This reduces memory and compute by **50–100×**, making video generation tractable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Raw video dimensions: T=120 frames, H=720, W=1280, C=3 channels
T, H, W, C = 120, 720, 1280, 3
raw_pixels = T * H * W * C
print(f"Raw video pixels:     {raw_pixels:,}  ({raw_pixels/1e6:.1f}M)")

# Latent space: 4x spatial downsampling, 4x temporal downsampling, 4 latent channels
latent_T, latent_H, latent_W, latent_C = T // 4, H // 8, W // 8, 4
latent_vals = latent_T * latent_H * latent_W * latent_C
compression_ratio = raw_pixels / latent_vals
print(f"Latent space values:  {latent_vals:,}  ({latent_vals/1e6:.2f}M)")
print(f"Compression ratio:    {compression_ratio:.0f}x")

# Storage for 100M videos (float32 = 4 bytes)
n_videos = 100_000_000
raw_tb   = n_videos * raw_pixels  * 4 / 1e12
latent_tb = n_videos * latent_vals * 4 / 1e12
print(f"\n100M videos — Raw:    {raw_tb:,.0f} TB")
print(f"100M videos — Latent: {latent_tb:,.0f} TB")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["Raw", "Latent"], [raw_pixels/1e6, latent_vals/1e6], color=["#e74c3c", "#2ecc71"])
axes[0].set_title("Values per Video (millions)"); axes[0].set_ylabel("Millions of values")
axes[1].bar(["Raw", "Latent"], [raw_tb, latent_tb], color=["#e74c3c", "#2ecc71"])
axes[1].set_title("Storage for 100M Videos (TB)"); axes[1].set_ylabel("Terabytes")
for ax in axes:
    ax.bar_label(ax.containers[0], fmt="%.1f")
plt.suptitle(f"LDM Compression: {compression_ratio:.0f}x fewer values", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## Temporal Attention

Spatial attention lets each pixel attend to every other pixel **within the same frame**. Temporal attention extends this so that features at a given spatial location attend to the **same location across all frames** — enabling the model to reason about motion.

**Spatial attention** (within one frame):
```
Frame t:  [p1, p2, p3, p4, p5, p6]  ← every patch attends to every other patch
               ↕   ↕   ↕   ↕   ↕
```

**Temporal attention** (across frames at one spatial location):
```
Location (h, w):   frame0 ──► frame1 ──► frame2 ──► ... ──► frameT
                      ↑_____________↑_____________↑  (full self-attention)
```

Each spatial position independently runs self-attention over its T feature vectors. This is efficient: instead of attending over T×H×W tokens at once, we do H×W independent attention ops over T tokens each.

Key insight: by alternating **spatial attention** (within frame) and **temporal attention** (across frames), the model builds a rich spatiotemporal representation without the O((T·H·W)²) cost of full 3D attention.

In [ ]:
import torch
import torch.nn as nn

class TemporalAttention(nn.Module):
    """Self-attention across the time dimension for a video feature tensor."""
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x: (B, T, H, W, C)
        B, T, H, W, C = x.shape
        # Merge spatial dims into batch: each (h,w) location attends over T frames
        x_flat = x.permute(0, 2, 3, 1, 4).reshape(B * H * W, T, C)  # (B*H*W, T, C)
        # Self-attention across T
        out, _ = self.attn(x_flat, x_flat, x_flat)                   # (B*H*W, T, C)
        out = self.norm(out + x_flat)                                 # residual + norm
        # Restore original shape
        out = out.reshape(B, H, W, T, C).permute(0, 3, 1, 2, 4)      # (B, T, H, W, C)
        return out

# Quick smoke test
B, T, H, W, C = 1, 8, 4, 4, 64
layer = TemporalAttention(dim=C, num_heads=4)
x = torch.randn(B, T, H, W, C)
y = layer(x)
print(f"Input:  {tuple(x.shape)}")
print(f"Output: {tuple(y.shape)}  (shape preserved)")
print(f"Params: {sum(p.numel() for p in layer.parameters()):,}")

In [ ]:
import torch
import torch.nn as nn

# 2D Conv: processes one frame at a time — no cross-frame information
conv2d = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), padding=1)

# 3D Conv: kernel (kT, kH, kW) spans time AND space — captures motion patterns
conv3d = nn.Conv3d(in_channels=64, out_channels=64, kernel_size=(3, 3, 3), padding=1)

# Demonstrate on a video batch: (B, C, T, H, W)
B, C, T, H, W = 2, 64, 8, 16, 16
video = torch.randn(B, C, T, H, W)

# 2D conv must process each frame independently
frame = video[:, :, 0, :, :]          # (B, C, H, W) — single frame
out2d = conv2d(frame)

# 3D conv processes the full video, mixing temporal neighbors
out3d = conv3d(video)

print("3D Conv kernel (kT=3): each output pixel sees 3 consecutive frames")
print(f"  2D Conv input/output: {tuple(frame.shape)} -> {tuple(out2d.shape)}")
print(f"  3D Conv input/output: {tuple(video.shape)} -> {tuple(out3d.shape)}")
print(f"  2D params: {sum(p.numel() for p in conv2d.parameters()):,}")
print(f"  3D params: {sum(p.numel() for p in conv3d.parameters()):,}  (3x more due to kT=3)")

## 3D Patchify for Video DiT

Image DiT (Diffusion Transformer) slices a 2D image into non-overlapping patches. For video, we extend this to **spatiotemporal cubes** — small 3D blocks that span both space and time.

**Image patchify** (2D):
```
Image: (H, W, C)  →  patches: (N_patches, patch_H × patch_W × C)
  e.g. 256×256 image, patch 16×16 → 256 tokens of dim 768
```

**Video patchify** (3D):
```
Video: (T, H, W, C)  →  cubes: (N_cubes, patch_T × patch_H × patch_W × C)
  e.g. T=16, H=32, W=32, C=4 latent, patch (2,4,4)
       → (8 × 8 × 8) = 512 tokens of dim (2×4×4×4) = 128
```

**Why cubes?**
- Each token now encodes **local motion** (adjacent frames within the cube)
- Reduces sequence length: 512 tokens vs 32768 per-pixel tokens for the same clip
- The transformer then runs full attention over those 512 spatiotemporal tokens

This is exactly the approach used in **CogVideoX** and **Open-Sora**, where a 3D VAE first compresses video into latents and a 3D patchifier then tokenizes the latents for the DiT.

## Quiz

**Q1. Why do video generation models use a latent diffusion approach rather than running diffusion directly on pixels?**

<details>
<summary>Show answer</summary>

Raw video tensors are enormous (e.g., 110M values for a 5-second 720p clip). Running diffusion directly on pixels would require far more GPU memory and compute than is practical. A VAE first compresses the video into a latent space that is 50–100× smaller, making diffusion tractable while a decoder recovers visual quality at the end.
</details>

---

**Q2. In temporal attention, what does reshaping `(B, T, H, W, C)` → `(B×H×W, T, C)` achieve?**

<details>
<summary>Show answer</summary>

It merges the batch and spatial dimensions so that each spatial location `(h, w)` is treated as an independent sequence of T feature vectors. Standard multi-head self-attention then runs along the T dimension for every location in parallel — this is efficient because each attention operation is over T tokens (small) rather than T×H×W tokens (huge).
</details>

---

**Q3. What is the key advantage of 3D convolution over 2D convolution for video, and what is the cost?**

<details>
<summary>Show answer</summary>

A 3D conv kernel (kT, kH, kW) spans neighboring frames, so each output feature aggregates information from multiple time steps — capturing local motion patterns that 2D conv (which processes one frame at a time) cannot see. The cost is proportionally more parameters and FLOPs: a kernel size of (3,3,3) has 3× the parameters of (1,3,3), and the operation must hold the full video volume in memory.
</details>